# Radiografía del delito en Los Ángeles (2020–2023)
**Certamen 2 — Entrega final, Bloques 1 a 7** · Taller de Visualización de la Información (UDD)
Juan Pablo Castillo · Martín Cáceres · Emilio Pacheco

**Dataset:** *Crimes in Los Angeles 2020–2023* (Kaggle, asaniczka) · Fuente: LAPD · Licencia CC0 · Acceso: 06-10-2026.
Antes de correr: subir `Crime_Data_from_2020_to_Present.csv` a `/content/` (panel de archivos de Colab) y esperar a que termine de cargar.

In [ ]:
# Librerías clásicas (Clase 5)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("ggplot")
# Filtrar warnings
import warnings
warnings.filterwarnings("ignore")

## Bloque 1 — Perfil general del dataset: ¿qué datos tengo?
*Feedback del Avance 1:* sin observaciones de contenido; se mantiene el bloque.

In [ ]:
# Carga del dataset
ruta = "/content/Crime_Data_from_2020_to_Present.csv"
df = pd.read_csv(ruta)
df.shape  # (filas, columnas)

In [ ]:
# Tipos de datos, nulos por columna y memoria
df.info()

In [ ]:
# Muestra aleatoria de 5 registros
df.sample(5)

### Diccionario de variables
Fuente de las descripciones: diccionario oficial del LAPD (data.lacity.org). Ejemplos tomados del dataset.

| Variable | Tipo original | Descripción | Unidad | Ejemplo |
|---|---|---|---|---|
| division_number | int64 | Número de denuncia (identificador único; DR_NO en el LAPD) | — | 10304468 |
| date_reported | object | Fecha en que se hizo la denuncia | fecha | 2020-01-08 |
| date_occurred | object | Fecha y hora en que ocurrió el delito | fecha y hora | 2020-01-08 22:30:00 |
| area | int64 | Código del área policial (21 áreas) | — | 3 |
| area_name | object | Nombre del área policial | — | Southwest |
| reporting_district | int64 | Subdistrito de reporte dentro del área | — | 377 |
| part | int64 | Gravedad según el FBI (1 = más grave, 2 = menos grave) | — | 2 |
| crime_code | int64 | Código del delito | — | 624 |
| crime_description | object | Descripción del delito | — | BATTERY - SIMPLE ASSAULT |
| modus_operandi | object | Códigos del modo de operar del delincuente | — | 0444 0913 |
| victim_age | int64 | Edad de la víctima | años | 36 |
| victim_sex | object | Sexo de la víctima (F, M, X) | — | F |
| victim_descent | object | Código de origen étnico de la víctima | — | B |
| premise_code | float64 | Código del tipo de lugar | — | 501.0 |
| premise_description | object | Tipo de lugar donde ocurrió | — | SINGLE FAMILY DWELLING |
| weapon_code | float64 | Código del arma usada | — | 400.0 |
| weapon_description | object | Arma usada | — | STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE) |
| status | object | Código del estado del caso | — | AO |
| status_description | object | Estado del caso | — | Adult Other |
| crime_code_1 a crime_code_4 | float64 | Códigos de delitos adicionales del mismo hecho | — | 624.0 |
| location | object | Dirección (redondeada a la centena de cuadra) | — | 1100 W 39TH PL |
| cross_street | object | Calle transversal | — | — |
| latitude | float64 | Latitud | grados | 34.0141 |
| longitude | float64 | Longitud | grados | -118.2978 |

### Corrección de tipos
- **Fechas** (texto → `datetime`): permiten extraer hora, día y mes y calcular retrasos.
- **Identificador** `division_number` (número → texto): no es una cantidad.
- **Códigos** (número → `category`): son etiquetas; promediarlos no tiene sentido.
- **Textos repetidos** (object → `category`): pocos valores que se repiten (21 áreas, 138 delitos); ocupan menos memoria.
- Se mantienen numéricas `victim_age`, `latitude` y `longitude`, que sí son cantidades.

In [ ]:
# Fechas: texto -> datetime (Guía Netflix)
df["date_reported"] = pd.to_datetime(df["date_reported"])
df["date_occurred"] = pd.to_datetime(df["date_occurred"])

# Identificador: número -> texto (Control II)
df["division_number"] = df["division_number"].astype("str")

# Códigos: número -> category
df["area"] = df["area"].astype("category")
df["reporting_district"] = df["reporting_district"].astype("category")
df["part"] = df["part"].astype("category")
df["crime_code"] = df["crime_code"].astype("category")
df["premise_code"] = df["premise_code"].astype("category")
df["weapon_code"] = df["weapon_code"].astype("category")
df["crime_code_1"] = df["crime_code_1"].astype("category")
df["crime_code_2"] = df["crime_code_2"].astype("category")
df["crime_code_3"] = df["crime_code_3"].astype("category")
df["crime_code_4"] = df["crime_code_4"].astype("category")

# Textos repetidos: object -> category
df["status"] = df["status"].astype("category")
df["area_name"] = df["area_name"].astype("category")
df["crime_description"] = df["crime_description"].astype("category")
df["victim_sex"] = df["victim_sex"].astype("category")
df["victim_descent"] = df["victim_descent"].astype("category")
df["premise_description"] = df["premise_description"].astype("category")
df["weapon_description"] = df["weapon_description"].astype("category")
df["status_description"] = df["status_description"].astype("category")

In [ ]:
# Verificación: nuevos tipos y memoria (de 175,7+ MB a 78,2+ MB)
df.info()

## Bloque 2 — Calidad de los datos: ¿qué tan completos y confiables son?
*Feedback del Avance 2:* sin observaciones de contenido; se migraron todos los gráficos a Seaborn y se agregó la comparación antes/después.

In [ ]:
# Nulos por variable: conteo y porcentaje
nulos = df.isnull().sum()
nulos_pct = (df.isnull().mean() * 100).round(2)
tabla_nulos = pd.DataFrame({"Nulos": nulos, "Porcentaje": nulos_pct})
tabla_nulos = tabla_nulos[tabla_nulos["Nulos"] > 0].sort_values("Nulos", ascending=False)
tabla_nulos

In [ ]:
# Gráfico del % de nulos por variable (Seaborn)
plt.figure(figsize=(10, 5))
sns.barplot(x=tabla_nulos["Porcentaje"], y=tabla_nulos.index, color="#8B1E1E")
plt.title("Porcentaje de valores nulos por variable")
plt.xlabel("Nulos (% de los registros)")
plt.ylabel("Variable")
plt.show()

In [ ]:
# Registros duplicados
df.duplicated().sum()

In [ ]:
# Inconsistencias en sexo: el LAPD solo documenta F, M y X
df["victim_sex"].value_counts(dropna=False)

In [ ]:
# Edades imposibles o sospechosas
print("Edad = 0:", (df["victim_age"] == 0).sum())
print("Edad negativa:", (df["victim_age"] < 0).sum())
print("Edad máxima:", df["victim_age"].max())

# ¿Qué delitos tienen edad 0?
df[df["victim_age"] == 0]["crime_description"].value_counts().head(5)

In [ ]:
# Coordenadas inválidas: (0,0) es ubicación faltante según el LAPD; se revisan también nulos
print("Coordenadas (0,0):", ((df["latitude"] == 0) & (df["longitude"] == 0)).sum())
print("Coordenadas nulas:", df["latitude"].isnull().sum() + df["longitude"].isnull().sum())

# Horas por defecto: 12:00 y 00:01 exactas
print("A las 12:00:", ((df["date_occurred"].dt.hour == 12) & (df["date_occurred"].dt.minute == 0)).sum())
print("A las 00:01:", ((df["date_occurred"].dt.hour == 0) & (df["date_occurred"].dt.minute == 1)).sum())

### Decisiones de limpieza
| Problema | Decisión | Fundamento |
|---|---|---|
| Arma vacía (65,2%) | Rellenar con "Sin arma" | El vacío indica que no se usó arma |
| Sexo, origen y lugar vacíos | Rellenar con "Sin dato" | No se puede inferir; queda explícito |
| Sexo "H" y "-" | Reagrupar en "X" | No están en el diccionario del LAPD |
| Edad 0 o negativa | Excluir solo en el análisis de edad (`df_edad`) | Delitos sin una persona como víctima |
| Coordenadas (0,0) | Excluir solo en mapas (`df_mapa`) | El LAPD las define como ubicación faltante |

No se elimina ninguna columna.

In [ ]:
# Copias ANTES de limpiar, para la comparación antes/después
edad_antes = df["victim_age"].copy()
arma_antes = df["weapon_description"].copy()

# Reagrupar categorías no documentadas
df["victim_sex"] = df["victim_sex"].replace({"H": "X", "-": "X"})
df["victim_descent"] = df["victim_descent"].replace({"-": "X"})

# Nulos con significado -> categoría explícita
df["weapon_description"] = df["weapon_description"].cat.add_categories("Sin arma").fillna("Sin arma")
df["victim_sex"] = df["victim_sex"].cat.add_categories("Sin dato").fillna("Sin dato")
df["victim_descent"] = df["victim_descent"].cat.add_categories("Sin dato").fillna("Sin dato")
df["premise_description"] = df["premise_description"].cat.add_categories("Sin dato").fillna("Sin dato")

# Subconjuntos para análisis específicos
df_edad = df[df["victim_age"] > 0]                                 # solo víctimas persona
df_mapa = df[(df["latitude"] != 0) & (df["longitude"] != 0)]        # solo ubicación válida
print("df_edad:", df_edad.shape, "| df_mapa:", df_mapa.shape)

### Antes y después de la limpieza

In [ ]:
# Antes/después 1: edad con y sin los ceros (figura de dos paneles)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(x=edad_antes, bins=30, ax=axes[0], color="#6c7480")
axes[0].set_title("Antes: edad con ceros (media " + str(round(edad_antes.mean(), 1)) + ")")
axes[0].set_xlabel("Edad (años)")
axes[0].set_ylabel("Frecuencia (delitos)")
sns.histplot(x=df_edad["victim_age"], bins=30, ax=axes[1], color="#8B1E1E")
axes[1].set_title("Después: solo víctimas persona (media " + str(round(df_edad["victim_age"].mean(), 1)) + ")")
axes[1].set_xlabel("Edad (años)")
axes[1].set_ylabel("Frecuencia (delitos)")
plt.tight_layout()
plt.show()

In [ ]:
# Antes/después 2: arma con nulos vs con la categoría "Sin arma"
antes = pd.DataFrame({"Categoría": ["Arma registrada", "Vacío (NaN)"],
                      "Delitos": [arma_antes.notnull().sum(), arma_antes.isnull().sum()]})
despues = df["weapon_description"].astype("str").value_counts().head(5)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=antes, x="Delitos", y="Categoría", ax=axes[0], color="#6c7480")
axes[0].set_title("Antes: el vacío no tiene significado")
axes[0].set_xlabel("Cantidad de delitos")
axes[0].set_ylabel("")
sns.barplot(x=despues.values, y=despues.index, ax=axes[1], color="#8B1E1E")
axes[1].set_title("Después: 'Sin arma' es la categoría principal")
axes[1].set_xlabel("Cantidad de delitos")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

## Bloque 3 — Exploración univariada: ¿cómo se comporta cada variable?
*Feedback del Avance 2:* sin observaciones de contenido; histogramas, boxplots, KDE y barras pasan a Seaborn (`histplot`, `boxplot`, `kdeplot`, `countplot`).
Variable derivada: **retraso de la denuncia** = días entre el delito y su denuncia.

In [ ]:
# Variable continua derivada: días entre el hecho y la denuncia
df["retraso_dias"] = (df["date_reported"] - df["date_occurred"].dt.normalize()).dt.days
df["retraso_dias"].head()

In [ ]:
# Resumen de todas las variables numéricas (Clase 4)
df[["victim_age", "latitude", "longitude", "retraso_dias"]].describe().T.round(2)

In [ ]:
# Estadísticas de la edad (solo víctimas persona)
edad = df_edad["victim_age"]
p25 = edad.quantile(0.25)
p75 = edad.quantile(0.75)
print("Media:", round(edad.mean(), 2))
print("Mediana:", edad.median())
print("Moda:", edad.mode()[0])
print("Desviación estándar:", round(edad.std(), 2))
print("Mínimo:", edad.min(), "| Máximo:", edad.max())
print("P25:", p25, "| P75:", p75, "| RIC:", p75 - p25)
print("Asimetría:", round(edad.skew(), 2))

In [ ]:
# Edad: histograma, boxplot y densidad (Seaborn)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sns.histplot(data=df_edad, x="victim_age", bins=30, ax=axes[0], color="#8B1E1E")
axes[0].set_title("Histograma de la edad")
axes[0].set_xlabel("Edad (años)")
axes[0].set_ylabel("Frecuencia (víctimas)")
sns.boxplot(data=df_edad, x="victim_age", ax=axes[1], color="#8B1E1E")
axes[1].set_title("Boxplot de la edad")
axes[1].set_xlabel("Edad (años)")
sns.kdeplot(data=df_edad, x="victim_age", fill=True, ax=axes[2], color="#0b2545")
axes[2].set_title("Densidad (KDE) de la edad")
axes[2].set_xlabel("Edad (años)")
axes[2].set_ylabel("Densidad")
plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas del retraso de la denuncia
retraso = df["retraso_dias"]
q25 = retraso.quantile(0.25)
q75 = retraso.quantile(0.75)
print("Media:", round(retraso.mean(), 2))
print("Mediana:", retraso.median())
print("Moda:", retraso.mode()[0])
print("Desviación estándar:", round(retraso.std(), 2))
print("Mínimo:", retraso.min(), "| Máximo:", retraso.max())
print("P25:", q25, "| P75:", q75, "| RIC:", q75 - q25)
print("Asimetría:", round(retraso.skew(), 2))
print("% denunciado el mismo día:", round((retraso == 0).mean() * 100, 1))
print("Casos con más de un año:", (retraso > 365).sum())

In [ ]:
# Retraso: histograma (0 a 30 días, donde está la mayoría), boxplot y densidad
retraso_30 = df[df["retraso_dias"] <= 30]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sns.histplot(data=retraso_30, x="retraso_dias", bins=31, ax=axes[0], color="#8B1E1E")
axes[0].set_title("Retraso de la denuncia (0 a 30 días)")
axes[0].set_xlabel("Días entre el delito y la denuncia")
axes[0].set_ylabel("Frecuencia (delitos)")
sns.boxplot(data=df, x="retraso_dias", ax=axes[1], color="#8B1E1E")
axes[1].set_xlim(0, 30)  # solo se recorta la vista; el boxplot se calcula con todos los casos
axes[1].set_title("Boxplot del retraso (todos los casos, vista 0 a 30)")
axes[1].set_xlabel("Días")
sns.kdeplot(data=retraso_30, x="retraso_dias", fill=True, ax=axes[2], color="#0b2545")
axes[2].set_title("Densidad del retraso (0 a 30 días)")
axes[2].set_xlabel("Días")
axes[2].set_ylabel("Densidad")
plt.tight_layout()
plt.show()

In [ ]:
# Variables categóricas: categorías únicas y proporciones (%)
for col in ["crime_description", "premise_description", "weapon_description",
            "area_name", "victim_sex", "status_description"]:
    print(col, "| categorías únicas:", df[col].nunique())
    print((df[col].value_counts() / len(df) * 100).round(1).head(3))
    print()

In [ ]:
# Countplot top 10: tipo de delito, lugar y arma (Seaborn)
for col, titulo in [("crime_description", "Top 10 tipos de delito"),
                    ("premise_description", "Top 10 lugares del delito"),
                    ("weapon_description", "Top 10 armas (incluye 'Sin arma')")]:
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df, y=col, order=df[col].value_counts().head(10).index, color="#0b2545")
    plt.title(titulo + " (2020-2023)")
    plt.xlabel("Cantidad de delitos")
    plt.ylabel("")
    plt.show()

## Bloque 4 — Mapas: ¿dónde ocurre el delito?
*Feedback del Avance 2:* sin observaciones de contenido; los mapas se rehacen en Seaborn (`scatterplot` y `kdeplot`) con ejes en proporción y recuadro acotado.
Se usan los 850.687 delitos con coordenadas válidas (`df_mapa`): se excluyeron las 2.263 coordenadas (0,0) y no hay coordenadas nulas.

In [ ]:
# Muestra de 100.000 delitos: la pauta la pide sobre 100.000 puntos (tenemos 850.687)
muestra = df_mapa.sample(100000, random_state=42)

# ¿La muestra representa al total? Comparamos proporciones (puntos porcentuales)
print("Diferencia máxima por área:",
      round(((df_mapa["area_name"].value_counts(normalize=True) - muestra["area_name"].value_counts(normalize=True)).abs().max() * 100), 2), "pp")
print("Diferencia máxima por gravedad:",
      round(((df_mapa["part"].value_counts(normalize=True) - muestra["part"].value_counts(normalize=True)).abs().max() * 100), 2), "pp")

In [ ]:
# Mapa de puntos: muestra de 100.000 coloreada por gravedad (1 = grave, 2 = menos grave)
fig, ax = plt.subplots(figsize=(9, 9))
sns.scatterplot(data=muestra, x="longitude", y="latitude", hue="part",
                palette={1: "#8B1E1E", 2: "#0b2545"}, s=2, alpha=0.3, linewidth=0, ax=ax)
ax.set_aspect("equal")
ax.set_xlim(-118.70, -118.15)
ax.set_ylim(33.70, 34.35)
ax.set_title("Delitos por gravedad (muestra de 100.000, 2020-2023)")
ax.set_xlabel("Longitud (grados)")
ax.set_ylabel("Latitud (grados)")
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ["1 = grave", "2 = menos grave"], title="Gravedad (Part)", markerscale=6)
plt.show()

In [ ]:
# Mapa de densidad: kdeplot sobre la misma muestra de 100.000
fig, ax = plt.subplots(figsize=(9, 9))
sns.kdeplot(data=muestra, x="longitude", y="latitude", fill=True, cmap="YlOrRd",
            levels=10, thresh=0.05, ax=ax)
ax.set_aspect("equal")
ax.set_xlim(-118.70, -118.15)
ax.set_ylim(33.70, 34.35)
ax.set_title("Densidad de delitos (kdeplot, muestra de 100.000)")
ax.set_xlabel("Longitud (grados)")
ax.set_ylabel("Latitud (grados)")
plt.show()

In [ ]:
# Mapa de puntos: dos delitos de vehículos en 2023 (34.505 < 100.000, no requiere muestra)
dos = ["VEHICLE - STOLEN", "BURGLARY FROM VEHICLE"]
df_dos = df_mapa[(df_mapa["crime_description"].isin(dos)) &
                 (df_mapa["date_occurred"].dt.year == 2023)]
fig, ax = plt.subplots(figsize=(9, 9))
sns.scatterplot(data=df_dos, x="longitude", y="latitude", hue="crime_description",
                hue_order=dos, palette=["#0b2545", "#e07a6f"], s=3, alpha=0.3, linewidth=0, ax=ax)
ax.set_aspect("equal")
ax.set_xlim(-118.70, -118.15)
ax.set_ylim(33.70, 34.35)
ax.set_title("Robo de vehículo vs. robo desde vehículo (2023)")
ax.set_xlabel("Longitud (grados)")
ax.set_ylabel("Latitud (grados)")
ax.legend(title="Delito", markerscale=6)
plt.show()

In [ ]:
# Cifras para interpretar los mapas
print("Delitos por área (top 5 y mínimo):")
print(df_mapa["area_name"].value_counts().head(5))
print(df_mapa["area_name"].value_counts().tail(1))
print("\nGravedad (%):")
print((df_mapa["part"].value_counts(normalize=True) * 100).round(1))

robo_desde = df_mapa[df_mapa["crime_description"] == "BURGLARY FROM VEHICLE"]
robo_de = df_mapa[df_mapa["crime_description"] == "VEHICLE - STOLEN"]
print("\nRobo desde vehículo (% por área):")
print((robo_desde["area_name"].value_counts() / len(robo_desde) * 100).round(1).head(3))
print("\nRobo de vehículo (% por área):")
print((robo_de["area_name"].value_counts() / len(robo_de) * 100).round(1).head(3))

## Bloque 5 — Valores atípicos: ¿qué valores se salen de lo esperado?
Regla IQR: es atípico todo valor bajo Q1 − 1,5·RIC o sobre Q3 + 1,5·RIC. Se aplica a las variables numéricas: edad (`df_edad`), retraso de la denuncia, latitud y longitud (`df_mapa`) y hora del hecho.

In [ ]:
# Hora del hecho como variable numérica
df["hora"] = df["date_occurred"].dt.hour

# Regla IQR para cada variable numérica
variables = [("victim_age", df_edad), ("retraso_dias", df), ("latitude", df_mapa),
             ("longitude", df_mapa), ("hora", df)]
filas = []
for col, datos in variables:
    q1 = datos[col].quantile(0.25)
    q3 = datos[col].quantile(0.75)
    ric = q3 - q1
    lim_inf = q1 - 1.5 * ric
    lim_sup = q3 + 1.5 * ric
    n_atip = ((datos[col] < lim_inf) | (datos[col] > lim_sup)).sum()
    filas.append({"Variable": col, "Q1": q1, "Q3": q3, "RIC": ric,
                  "Límite inferior": lim_inf, "Límite superior": lim_sup,
                  "Atípicos": n_atip, "% atípicos": round(n_atip / len(datos) * 100, 1)})
tabla_atipicos = pd.DataFrame(filas)
tabla_atipicos.round(2)

In [ ]:
# Diagnóstico: ¿qué hay detrás de los atípicos?
print("Edad sobre 83 años, delitos más frecuentes:")
print(df_edad[df_edad["victim_age"] > 83]["crime_description"].value_counts().head(3))
print("\nVíctimas con edad 99:", (df_edad["victim_age"] == 99).sum())
print("\nRetraso sobre 5 días, delitos más frecuentes (%):")
print((df[df["retraso_dias"] > 5]["crime_description"].value_counts(normalize=True) * 100).round(1).head(3))
print("\nLatitud bajo 33,79 (franja sur), áreas:")
print(df_mapa[df_mapa["latitude"] < 33.79]["area_name"].value_counts().head(3))

**Decisión: se mantienen todos los atípicos.** Son valores reales del dominio: adultos mayores víctimas de robo a vivienda y de identidad, robos de identidad que se descubren semanas después y la franja sur de la ciudad (Harbor), que la regla IQR marca solo por la forma alargada de Los Ángeles. Eliminarlos borraría casos relevantes para la policía. El gráfico siguiente muestra qué se perdería si se eliminaran.

In [ ]:
# Boxplot antes/después: distribución con atípicos (decisión) vs. si se eliminaran
for col, datos, unidad in [("victim_age", df_edad, "Edad (años)"), ("retraso_dias", df, "Días")]:
    fila = tabla_atipicos[tabla_atipicos["Variable"] == col]
    lim_sup = fila["Límite superior"].values[0]
    sin_atipicos = datos[datos[col] <= lim_sup]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    sns.boxplot(data=datos, x=col, ax=axes[0], color="#8B1E1E")
    axes[0].set_title(col + ": con atípicos (se mantienen)")
    axes[0].set_xlabel(unidad)
    if col == "retraso_dias":
        axes[0].set_xlim(0, 30)  # vista 0 a 30 días; los casos sobre 30 siguen en los datos
    sns.boxplot(data=sin_atipicos, x=col, ax=axes[1], color="#6c7480")
    axes[1].set_title(col + ": si se eliminaran (" + str(len(datos) - len(sin_atipicos)) + " casos menos)")
    axes[1].set_xlabel(unidad)
    plt.tight_layout()
    plt.show()

## Bloque 6 — Exploración temporal: ¿cómo cambian los datos en el tiempo?
Se excluye diciembre de 2023 (los datos terminan el 4 de diciembre: 1.545 delitos) para no mostrar una caída falsa. Las 12:00 se señalan como hora por defecto.

In [ ]:
# Variables de tiempo y corte en noviembre de 2023
df["anio"] = df["date_occurred"].dt.year
df["mes"] = df["date_occurred"].dt.month
df_tiempo = df[~((df["anio"] == 2023) & (df["mes"] == 12))]
print("Diciembre 2023 excluido:", len(df) - len(df_tiempo), "delitos")

# Delitos por año y mes
mensual = df_tiempo.groupby(["anio", "mes"]).size().reset_index(name="delitos")
mensual.sort_values("delitos").head(3)

In [ ]:
# Lineplot: delitos por mes, una línea por año
plt.figure(figsize=(11, 5))
sns.lineplot(data=mensual, x="mes", y="delitos", hue="anio", marker="o", palette=["#6c7480", "#0b2545", "#e07a6f", "#8B1E1E"])
plt.title("Delitos por mes y año (2020 - noviembre 2023)")
plt.xlabel("Mes")
plt.ylabel("Cantidad de delitos")
plt.xticks(range(1, 13))
plt.legend(title="Año")
plt.show()

In [ ]:
# Barplot: delitos por día de la semana
dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df_tiempo["dia"] = df_tiempo["date_occurred"].dt.day_name()
por_dia = df_tiempo["dia"].value_counts().reindex(dias)
plt.figure(figsize=(10, 4.5))
sns.barplot(x=por_dia.index, y=por_dia.values, color="#0b2545")
plt.title("Delitos por día de la semana (2020 - noviembre 2023)")
plt.xlabel("Día de la semana")
plt.ylabel("Cantidad de delitos")
plt.show()
por_dia

In [ ]:
# Barplot: delitos por hora, con las 12:00 señaladas (hora por defecto)
por_hora = df_tiempo["hora"].value_counts().sort_index()
colores = ["#e07a6f" if h == 12 else "#0b2545" for h in por_hora.index]
plt.figure(figsize=(12, 4.5))
sns.barplot(x=por_hora.index, y=por_hora.values, palette=colores)
plt.title("Delitos por hora del día (en rojo claro: 12:00, inflada por la hora por defecto)")
plt.xlabel("Hora del día (0-23)")
plt.ylabel("Cantidad de delitos")
plt.show()

## Bloque 7 — Relaciones entre variables: ¿qué variables se mueven juntas?
Numéricas: edad, retraso, hora, latitud y longitud (víctimas persona con ubicación válida). Grupos: los 5 delitos más frecuentes.

In [ ]:
# Base para relaciones numéricas: víctimas persona con ubicación válida
base = df[(df["victim_age"] > 0) & (df["latitude"] != 0) & (df["longitude"] != 0)]
numericas = ["victim_age", "retraso_dias", "hora", "latitude", "longitude"]

# Heatmap de correlación (Pearson y Spearman)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(base[numericas].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Correlación de Pearson")
sns.heatmap(base[numericas].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Correlación de Spearman")
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot sobre una muestra de 5.000 casos (con todos, los puntos se tapan entre sí)
muestra_pp = base.sample(5000, random_state=42)
sns.pairplot(muestra_pp, vars=numericas, plot_kws={"s": 5, "alpha": 0.3})
plt.show()

In [ ]:
# Top 5 delitos para comparar grupos
top5 = df["crime_description"].value_counts().head(5).index
df_top5 = df[df["crime_description"].isin(top5)]
df_top5["crime_description"] = df_top5["crime_description"].astype("str")

# Boxplot: retraso de la denuncia por tipo de delito (eje recortado a 0-30 días para leerlo)
plt.figure(figsize=(11, 5))
sns.boxplot(data=df_top5, x="retraso_dias", y="crime_description", color="#8B1E1E")
plt.xlim(0, 30)
plt.title("Retraso de la denuncia por tipo de delito (top 5)")
plt.xlabel("Días entre el delito y la denuncia (vista 0-30)")
plt.ylabel("")
plt.show()
df_top5.groupby("crime_description")["retraso_dias"].median()

In [ ]:
# Violin plot: edad de la víctima por tipo de delito (solo víctimas persona)
edad_top5 = df_top5[df_top5["victim_age"] > 0]
plt.figure(figsize=(11, 5))
sns.violinplot(data=edad_top5, x="victim_age", y="crime_description", color="#0b2545")
plt.title("Edad de la víctima por tipo de delito (top 5)")
plt.xlabel("Edad (años)")
plt.ylabel("")
plt.show()
edad_top5.groupby("crime_description")["victim_age"].median()

In [ ]:
# Heatmap de tabla cruzada: tipo de delito x sexo de la víctima (% por fila)
tabla_sexo = pd.crosstab(df_top5["crime_description"], df_top5["victim_sex"].astype("str"), normalize="index") * 100
plt.figure(figsize=(9, 4.5))
sns.heatmap(tabla_sexo, annot=True, fmt=".0f", cmap="Blues")
plt.title("Sexo de la víctima por tipo de delito (% de cada fila)")
plt.xlabel("Sexo de la víctima")
plt.ylabel("")
plt.show()

In [ ]:
# Heatmap de tabla cruzada: tipo de delito x área policial (% por fila)
tabla_area = pd.crosstab(df_top5["crime_description"], df_top5["area_name"].astype("str"), normalize="index") * 100
plt.figure(figsize=(16, 4.5))
sns.heatmap(tabla_area, annot=True, fmt=".0f", cmap="Reds")
plt.title("Distribución de cada delito por área policial (% de cada fila)")
plt.xlabel("Área policial")
plt.ylabel("")
plt.show()